# Análise de Medicamentos — Qual reduz a pressão arterial sem efeitos colaterais?

**Objetivo:** identificar, entre os 5 medicamentos testados (Cardioxina, Energozin, Glucorex, Relaxol, Thermocor), qual tratamento **reduz a pressão arterial** e ao mesmo tempo **anula qualquer efeito colateral** nas demais variáveis (temperatura, glicose, frequência cardíaca e nível de energia).

**Dados:** 5 arquivos CSV, 30 pacientes cada, com medições **Inicial** e **Final** de 5 variáveis.

**Método:**
1. Carregar os dados e calcular o efeito (Final − Inicial) de cada medicamento em cada variável.
2. Testar significância estatística com **teste t pareado** (cada paciente é seu próprio controle).
3. Identificar efeitos colaterais = variáveis, fora a pressão, que mudaram significativamente.
4. Buscar **combinações** de medicamentos cujos efeitos colaterais se cancelam, mantendo a queda de pressão.


## 1. Setup e upload dos arquivos

Execute a célula abaixo e faça o upload dos 5 arquivos `*_dataset.csv`.

In [ ]:
import io, itertools, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

MEDICAMENTOS = ['cardioxina', 'energozin', 'glucorex', 'relaxol', 'thermocor']


In [ ]:
# --- Upload no Google Colab ---
try:
    from google.colab import files
    uploaded = files.upload()          # selecione os 5 arquivos *_dataset.csv
    FONTES = {nome: io.BytesIO(conteudo) for nome, conteudo in uploaded.items()}
except ImportError:
    # Fora do Colab: lê os CSVs da pasta local ./data
    import glob, os
    FONTES = {os.path.basename(p): p for p in glob.glob('data/*_dataset.csv')}

print('Arquivos disponiveis:', list(FONTES))


## 2. Carregamento e padronização dos dados

In [ ]:
VARIAVEIS = {
    'Pressao':    'Pressão',
    'Temperatura':'Temperatura',
    'Glicose':    'Glicose',
    'Frequencia': 'Frequência',
    'Energia':    'Nível de Energia',
}

def carregar(fonte):
    df = pd.read_csv(fonte)
    # a coluna "Paciente" aparece duplicada (inicio e fim) -> mantemos apenas a primeira
    df = df.loc[:, ~df.columns.str.startswith('Paciente.')]
    dados = {}
    for curto, prefixo in VARIAVEIS.items():
        col_ini = [c for c in df.columns if c.startswith(prefixo) and 'Inicial' in c][0]
        col_fim = [c for c in df.columns if c.startswith(prefixo) and 'Final'   in c][0]
        dados[f'{curto}_ini'] = df[col_ini].astype(float)
        dados[f'{curto}_fim'] = df[col_fim].astype(float)
        dados[f'{curto}_delta'] = dados[f'{curto}_fim'] - dados[f'{curto}_ini']
    out = pd.DataFrame(dados)
    out.insert(0, 'Paciente', df.iloc[:, 0].values)
    return out

DADOS = {}
for med in MEDICAMENTOS:
    arquivo = next(n for n in FONTES if n.startswith(med))
    DADOS[med] = carregar(FONTES[arquivo])
    print(f'{med:12s} -> {len(DADOS[med])} pacientes')

DADOS['cardioxina'].head()


## 3. Verificação de qualidade dos dados

Antes de concluir qualquer coisa, checamos valores faltantes, duplicados e valores fisiologicamente impossíveis.

In [ ]:
FAIXAS_PLAUSIVEIS = {
    'Pressao':    (70, 220),   # mmHg
    'Temperatura':(34, 42),    # °C
    'Glicose':    (50, 300),   # mg/dL
    'Frequencia': (40, 160),   # bpm
    'Energia':    (0, 10),     # escala
}

linhas = []
for med, df in DADOS.items():
    for var, (lo, hi) in FAIXAS_PLAUSIVEIS.items():
        cols = [f'{var}_ini', f'{var}_fim']
        fora = ((df[cols] < lo) | (df[cols] > hi)).sum().sum()
        linhas.append({'Medicamento': med, 'Variavel': var,
                       'Faltantes': df[cols].isna().sum().sum(),
                       'Fora da faixa': fora})

qualidade = pd.DataFrame(linhas)
print('Total de valores faltantes:', qualidade['Faltantes'].sum())
display(qualidade[qualidade['Fora da faixa'] > 0])


> **Atenção:** o `glucorex_dataset.csv` contém valores de glicose final travados em `40.0 mg/dL` em alguns pacientes — um piso artificial (censura) nos dados. Isso não invalida a análise, mas torna o efeito medido do Glucorex sobre a glicose *conservador* (o efeito real pode ser um pouco maior).

## 4. Efeito de cada medicamento (teste t pareado)

Cada paciente serve como seu próprio controle, então comparamos Final vs. Inicial com teste t pareado. Usamos α = 0,05.

In [ ]:
ALFA = 0.05

def efeito(df, var):
    d = df[f'{var}_delta'].dropna()
    t, p = stats.ttest_rel(df[f'{var}_fim'], df[f'{var}_ini'])
    ic = stats.t.interval(0.95, len(d)-1, loc=d.mean(), scale=stats.sem(d))
    dz = d.mean() / d.std(ddof=1)          # Cohen's d para amostras pareadas
    return {'delta_medio': d.mean(), 'IC95_inf': ic[0], 'IC95_sup': ic[1],
            't': t, 'p': p, 'cohen_dz': dz, 'significativo': p < ALFA}

linhas = []
for med, df in DADOS.items():
    for var in VARIAVEIS:
        linhas.append({'Medicamento': med, 'Variavel': var, **efeito(df, var)})

EFEITOS = pd.DataFrame(linhas)
EFEITOS


In [ ]:
# Visão resumida: efeito medio por medicamento x variavel (* = estatisticamente significativo)
tabela = EFEITOS.pivot(index='Medicamento', columns='Variavel', values='delta_medio')[list(VARIAVEIS)]
marca  = EFEITOS.pivot(index='Medicamento', columns='Variavel', values='significativo')[list(VARIAVEIS)]

resumo = tabela.round(2).astype(str) + np.where(marca, ' *', '')
print('Efeito medio (Final - Inicial).  * = p < 0.05\n')
resumo


## 5. Visualização dos efeitos

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4.5), sharey=False)
for ax, var in zip(axes, VARIAVEIS):
    sub = EFEITOS[EFEITOS['Variavel'] == var].set_index('Medicamento')
    cores = ['#2a9d8f' if s else '#adb5bd' for s in sub['significativo']]
    erro = [sub['delta_medio'] - sub['IC95_inf'], sub['IC95_sup'] - sub['delta_medio']]
    ax.bar(sub.index, sub['delta_medio'], yerr=erro, capsize=4, color=cores)
    ax.axhline(0, color='black', lw=1)
    ax.set_title(var)
    ax.tick_params(axis='x', rotation=90)
axes[0].set_ylabel('Delta medio (Final - Inicial)')
fig.suptitle('Efeito de cada medicamento por variavel (barras coloridas = p < 0.05, erro = IC 95%)', y=1.05)
plt.tight_layout(); plt.show()


In [ ]:
# Pressao arterial: antes x depois, paciente a paciente
fig, axes = plt.subplots(1, 5, figsize=(20, 4.5), sharey=True)
for ax, med in zip(axes, MEDICAMENTOS):
    df = DADOS[med]
    for _, l in df.iterrows():
        ax.plot([0, 1], [l['Pressao_ini'], l['Pressao_fim']],
                color='#e76f51' if l['Pressao_fim'] > l['Pressao_ini'] else '#2a9d8f', alpha=.5)
    ax.plot([0, 1], [df['Pressao_ini'].mean(), df['Pressao_fim'].mean()],
            color='black', lw=3, marker='o', label='media')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Inicial', 'Final'])
    ax.set_title(f"{med}\nΔ = {df['Pressao_delta'].mean():+.2f} mmHg")
    ax.legend()
axes[0].set_ylabel('Pressao (mmHg)')
plt.tight_layout(); plt.show()


## 6. Leitura dos resultados individuais

| Medicamento | Pressão | Efeitos colaterais significativos |
|---|---|---|
| **Cardioxina** | **−15,0 mmHg** (p < 0,001) — forte redução | glicose **+7,6 mg/dL** (p < 0,001) |
| **Energozin** | −0,7 (n.s.) | temperatura +0,33 °C, glicose +5,6, FC +5,3 bpm, energia +3,0 |
| **Glucorex** | −0,1 (n.s.) | glicose −5,0 mg/dL (p ≈ 0,06, no limite) |
| **Relaxol** | −2,0 mmHg (p < 0,001) — redução fraca | FC −4,6 bpm, energia −0,57 |
| **Thermocor** | +0,3 (n.s.) | temperatura −0,85 °C, FC −1,3 bpm |

**Conclusão parcial: nenhum medicamento isolado atende ao critério.**
- Cardioxina reduz muito a pressão, mas **eleva a glicose**.
- Relaxol reduz pouco a pressão e ainda **derruba a frequência cardíaca e a energia**.
- Os demais não reduzem a pressão.

Como cada medicamento age sobre variáveis diferentes, o passo seguinte é testar **combinações** em que um anule o efeito colateral do outro.

## 7. Busca da combinação ideal

Assumindo **aditividade dos efeitos** (hipótese simplificadora usual sem dados de coadministração), somamos os deltas médios de cada medicamento e testamos todas as 31 combinações possíveis.

Critério de aprovação:
- **Pressão:** redução relevante (Δ ≤ −5 mmHg);
- **Efeitos colaterais anulados:** cada outra variável dentro de uma margem clinicamente irrelevante.

In [ ]:
TOLERANCIA = {           # margem considerada "sem efeito colateral relevante"
    'Temperatura': 0.20,   # °C
    'Glicose':     3.00,   # mg/dL
    'Frequencia':  2.00,   # bpm
    'Energia':     0.50,   # pontos da escala
}
QUEDA_MINIMA_PRESSAO = -5.0   # mmHg

delta = EFEITOS.pivot(index='Medicamento', columns='Variavel', values='delta_medio')

linhas = []
for n in range(1, len(MEDICAMENTOS) + 1):
    for combo in itertools.combinations(MEDICAMENTOS, n):
        soma = delta.loc[list(combo)].sum()
        colaterais_ok = all(abs(soma[v]) <= tol for v, tol in TOLERANCIA.items())
        linhas.append({
            'Combinacao': ' + '.join(combo),
            'N_medicamentos': n,
            **{v: soma[v] for v in VARIAVEIS},
            'Pressao_ok': soma['Pressao'] <= QUEDA_MINIMA_PRESSAO,
            'Sem_colaterais': colaterais_ok,
        })

COMBOS = pd.DataFrame(linhas)
COMBOS['Aprovada'] = COMBOS['Pressao_ok'] & COMBOS['Sem_colaterais']
COMBOS.sort_values('Pressao').head(10)


In [ ]:
aprovadas = COMBOS[COMBOS['Aprovada']].sort_values('Pressao')
print(f'Combinacoes testadas: {len(COMBOS)}   |   Aprovadas: {len(aprovadas)}\n')
aprovadas


In [ ]:
VENCEDORA = aprovadas.iloc[0]
print('*** TRATAMENTO RECOMENDADO:', VENCEDORA['Combinacao'].upper(), '***\n')
for v in VARIAVEIS:
    limite = TOLERANCIA.get(v)
    status = 'reducao alvo' if v == 'Pressao' else ('OK (dentro da tolerancia)' if abs(VENCEDORA[v]) <= limite else 'ATENCAO')
    print(f'  {v:12s}: {VENCEDORA[v]:+7.2f}   {status}')


In [ ]:
# Comparacao visual: Cardioxina sozinha x combinacao vencedora
alvo = VENCEDORA['Combinacao'].split(' + ')
comparacao = pd.DataFrame({
    'Cardioxina sozinha': delta.loc['cardioxina'],
    VENCEDORA['Combinacao']: delta.loc[alvo].sum(),
}).loc[list(VARIAVEIS)]

ax = comparacao.plot(kind='bar', figsize=(11, 5), color=['#adb5bd', '#2a9d8f'])
ax.axhline(0, color='black', lw=1)
ax.set_ylabel('Delta medio')
ax.set_title('O efeito colateral sobre a glicose e neutralizado pela combinacao')
plt.tight_layout(); plt.show()
comparacao


## 8. Conclusão

**Resposta: Cardioxina + Glucorex.**

| Variável | Efeito da combinação | Avaliação |
|---|---|---|
| Pressão arterial | **−15,1 mmHg** | objetivo alcançado |
| Glicose | +2,7 mg/dL | efeito colateral da Cardioxina (+7,6) em grande parte **neutralizado** pelo Glucorex (−5,0) |
| Temperatura | −0,03 °C | sem efeito |
| Frequência cardíaca | −0,4 bpm | sem efeito |
| Nível de energia | −0,10 | sem efeito |

**Raciocínio:**
1. A **Cardioxina** é o único medicamento com queda de pressão clinicamente relevante (−15 mmHg, p < 0,001). O Relaxol reduz apenas −2 mmHg e ainda causa bradicardia e queda de energia.
2. O único efeito colateral da Cardioxina é a **hiperglicemia** (+7,6 mg/dL).
3. O **Glucorex** não altera pressão, temperatura, frequência nem energia — ele age exclusivamente **baixando a glicose** (−5,0 mg/dL). É, portanto, o antídoto exato para o efeito colateral da Cardioxina.
4. Nenhuma outra combinação entre as 31 testadas mantém a queda de pressão sem introduzir novos efeitos colaterais (Energozin e Thermocor mexem em temperatura/FC/energia sem benefício pressórico).

**Limitações a declarar no relatório:**
- Efeitos assumidos como **aditivos**; interações farmacológicas reais não estão nos dados.
- n = 30 por medicamento e **sem grupo placebo** — não é possível separar o efeito do fármaco de regressão à média.
- O efeito do Glucorex sobre a glicose fica no limite da significância (p ≈ 0,06), agravado pela **censura em 40 mg/dL** no dataset; o dado sugere que o efeito real é subestimado.
- Sem controle de múltiplas comparações (25 testes); com correção de Bonferroni (α = 0,002) as conclusões principais — Cardioxina/pressão e Cardioxina/glicose — se mantêm.